# 📓 Session 5 · Exercises
### Fitting a model, and choosing between models

This is the first set of exercises where you fit something. The fitting itself is three lines, so most of the work here is the part that surrounds it: scoring against a baseline, comparing candidates without spending the test rows, and cross-validating with folds that respect the calendar.

You will build the volatility table from the lecture, fit a linear regression to it, beat two rules that use no model at all, watch a single validation split give two different answers, and finish by running the whole workflow on a stock the lecture never touched.

## How to use this notebook

- Run the **setup cell** below first. It loads the price data, builds the table the lecture used, and imports the three scikit-learn pieces.
- Each exercise has a **task**, then a **code cell** for your work. Cells with `...` are blanks to fill in. Replace them with real code.
- Stuck? Open the **💡 Hint**, but only after a genuine attempt. Open the **✅ Solution** to *check* yourself, not to skip the thinking.
- Every cell runs cleanly even with the blanks still in place, so pressing **Run all** never floods you with errors.
- Exercises do not depend on each other. If one defeats you, move on.

**You are not expected to finish all of these.** Do what you can, and come back to the rest when you revise. Short on time? Read the hint, then the solution. A worked solution you genuinely understand is real learning too.

**Returns are in percent here**, exactly as in the lecture, so an error of `0.41` means 0.41 percentage points.

### Difficulty

| badge | what to expect |
|:--|:--|
| ★☆☆☆☆ | One step, straight from the lecture. You are checking that you can type it. |
| ★★☆☆☆ | The same idea on new data, or two steps in a row. Nothing to decide. |
| ★★★☆☆ | Combine two ideas, or adapt a pattern rather than copy it. |
| ★★★★☆ | You choose the approach. Several steps, and something has to be worked out before you type. |
| ★★★★★ | A genuine puzzle: an insight, or a constraint that rules out the obvious route. Always solvable with what you have. |

The stars rate the work against **this** session. A three-star task here assumes everything from Sessions 1 to 4, so it is a bigger piece of work than a three-star task in an earlier notebook.

Some exercises also carry a **revisits** tag. Those need something from an earlier session as well as today's material, and they are there on purpose: the skills are meant to accumulate.

## 🧰 Your toolkit for today

Everything from Sessions 1 to 4 still applies. This card holds what Session 5 added.

> Names in brackets (`frame`, `columns`, `model`, ...) are **placeholders**: put your own variable there. **Hover any tool** to see what it does.

<p style="line-height:2.1"><strong>Building X and y</strong><br>
<code style="cursor:help" title="A LIST of column names inside the brackets, so the result stays a table. This is what X has to be.">frame[['col_a', 'col_b']]</code> &nbsp;&nbsp; <code style="cursor:help" title="One pair of brackets gives a single column. That is what y has to be.">frame['target']</code> &nbsp;&nbsp; <code style="cursor:help" title="Rows up to a date, and rows from a date. A table with a date index slices with dates.">frame.loc[:'2022-12-31']  ·  frame.loc['2023-01-01':]</code> &nbsp;&nbsp; <code style="cursor:help" title="Rows and columns, as a pair. Handy for checking a split went where you meant.">frame.shape</code></p>

<p style="line-height:2.1"><strong>Fitting and predicting</strong><br>
<code style="cursor:help" title="Create a model. It knows nothing until you fit it.">LinearRegression()</code> &nbsp;&nbsp; <code style="cursor:help" title="Read the training rows and compute the coefficients. Features first, target second.">model.fit(X, y)</code> &nbsp;&nbsp; <code style="cursor:help" title="One prediction per row of X, in the same order, as a plain numpy array.">model.predict(X)</code> &nbsp;&nbsp; <code style="cursor:help" title="The constant term the fit found. The trailing underscore means it came from the data.">model.intercept_</code> &nbsp;&nbsp; <code style="cursor:help" title="One coefficient per feature, in the order the columns came in.">model.coef_</code></p>

<p style="line-height:2.1"><strong>Scoring</strong><br>
<code style="cursor:help" title="Mean squared error. The TRUE values go first and the predictions second.">mean_squared_error(y_true, y_pred)</code> &nbsp;&nbsp; <code style="cursor:help" title="Square root, which puts a mean squared error back into the units of the target.">np.sqrt(value)</code> &nbsp;&nbsp; <code style="cursor:help" title="An array of one value repeated. A baseline that ignores the features looks like this.">np.full(n, value)</code></p>

<p style="line-height:2.1"><strong>Cross-validation</strong><br>
<code style="cursor:help" title="Folds that move forward in time: each scored block comes after the rows fitted on.">TimeSeriesSplit(n_splits=5)</code> &nbsp;&nbsp; <code style="cursor:help" title="Keep only the most recent rows for fitting, so old years are forgotten.">TimeSeriesSplit(n_splits=5, max_train_size=500)</code> &nbsp;&nbsp; <code style="cursor:help" title="Throw away the rows whose target reaches into the block about to be scored.">TimeSeriesSplit(n_splits=5, gap=20)</code> &nbsp;&nbsp; <code style="cursor:help" title="The textbook folds: the rows are shuffled first, which assumes they are interchangeable.">KFold(n_splits=5, shuffle=True, random_state=0)</code></p>

<p style="line-height:2.1"><strong>Running the folds</strong><br>
<code style="cursor:help" title="Fit and score once per fold, and hand back one number per fold.">cross_val_score(model, X, y, cv=folds, scoring='neg_root_mean_squared_error')</code> &nbsp;&nbsp; <code style="cursor:help" title="scikit-learn reports scores so that larger is better, so errors come back negative. Flip the sign.">-scores</code> &nbsp;&nbsp; <code style="cursor:help" title="The row positions of the fitted rows and the scored rows, one pair per fold.">folds.split(frame)</code></p>

**Formulas you will reach for**

| what | formula |
|:--|:--|
| Root mean squared error | $$\text{RMSE}=\sqrt{\dfrac{1}{n}\sum_i (y_i-\hat{y}_i)^2}$$ |
| Out-of-sample R squared | $$R^2=1-\dfrac{\sum_i (y_i-\hat{y}_i)^2}{\sum_i (y_i-\bar{y}_{\text{train}})^2}$$ |
| Cross-validation error | $$\text{CV}_K=\dfrac{1}{K}\sum_{k=1}^{K} e_k$$ |
| AIC | $$n\log(\text{MSE})+2d$$ |
| BIC | $$n\log(\text{MSE})+d\log(n)$$ |


---

## ⚙️ Setup: run this first

This loads the price data, rebuilds the table the lecture worked on, and imports the scikit-learn pieces. If you are in Google Colab it downloads the data by itself.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import cross_val_score, KFold, TimeSeriesSplit

CANDIDATE_DIRS = ["data", os.path.join("..", "data"), "."]
REPO_RAW_URL = "https://raw.githubusercontent.com/theill95/mlfin-2026/main/data/"   # used when the CSV files are not next to the notebook


def load_csv(filename, **kwargs):
    """Read one of the course CSV files, wherever it happens to be."""
    for folder in CANDIDATE_DIRS:
        path = os.path.join(folder, filename)
        if os.path.exists(path):
            return pd.read_csv(path, **kwargs)
    if REPO_RAW_URL is not None:
        return pd.read_csv(REPO_RAW_URL + filename, **kwargs)
    raise FileNotFoundError(
        f"Could not find {filename}. Run this notebook from the course folder, "
        f"upload the CSV into Colab, or set REPO_RAW_URL."
    )


# Eleven instruments, 2015 to 2024. Returns in PERCENT, as in the lecture.
prices = load_csv("prices.csv", parse_dates=["date"])
wide = prices.pivot(index="date", columns="ticker", values="close")
rets = wide.pct_change().dropna() * 100

# The lecture's table: three features looking back, one target looking forward
apple = rets["AAPL"]
table = pd.DataFrame({
    "vol_20d": apple.rolling(20).std(),
    "vol_60d": apple.rolling(60).std(),
    "ret_20d": apple.rolling(20).mean(),
})
table["vol_next"] = apple.rolling(20).std().shift(-20)
table = table.dropna()

train = table.loc[:"2022-12-31"]
test = table.loc["2023-01-01":]

# The three candidate feature sets from the lecture
A = ["vol_20d"]
B = ["vol_20d", "vol_60d"]
C = ["vol_20d", "vol_60d", "ret_20d"]

# For one exercise that needs a column with nothing in it
rng = np.random.default_rng(0)

print("table:", table.shape, "rows x columns")
print("train:", len(train), " test:", len(test))
print("columns:", list(table.columns))

---

## 🧱 A · The table, and the split

The setup cell already built these, but building them yourself once is the only way the rest of the notebook stops being magic.

### A1 · Apple's returns  ★☆☆☆☆

Take Apple's daily returns out of `rets` into a Series called `apple_ret`.

In [ ]:
apple_ret = ...
apple_ret

<details>
<summary>💡 Hint</summary>

`rets` has one column per ticker. Select a column by name.

</details>

<details>
<summary>✅ Solution</summary>

```python
apple_ret = rets['AAPL']
apple_ret
```

A return for every trading day, in percent. There are 2,515 of them.

</details>

---

### A2 · A feature that looks back  ★★☆☆☆  · revisits S4

Build `vol_20d`: the standard deviation of the **last twenty** daily returns, as a Series.

In [ ]:
vol_20d = ...
vol_20d

<details>
<summary>💡 Hint</summary>

`s.rolling(20).std()` gives a twenty-row window ending on the current row.

</details>

<details>
<summary>✅ Solution</summary>

```python
vol_20d = rets['AAPL'].rolling(20).std()
vol_20d
```

The window ends on the current row, so this feature only ever uses today and the nineteen days before it. The first nineteen values are `NaN`.

</details>

---

### A3 · A target that looks forward  ★★★☆☆  · revisits S4

Build `vol_next`: the volatility of the **next twenty** trading days, on today's row.

$$\text{vol\_next}_t = \text{sd}\big(r_{t+1}, \ldots, r_{t+20}\big)$$

In [ ]:
vol_next = ...
vol_next

<details>
<summary>💡 Hint 1</summary>

Start from the same rolling standard deviation as A2. That gives the twenty days **ending** today.

</details>

<details>
<summary>💡 Hint 2</summary>

`shift(-20)` moves values up by twenty rows, so the window that ends twenty days from now lands on today's row.

</details>

<details>
<summary>✅ Solution</summary>

```python
vol_next = rets['AAPL'].rolling(20).std().shift(-20)
vol_next
```

Backwards for the feature, forwards for the target, and the two windows never overlap. Getting the sign of the shift wrong is the most expensive mistake in this whole notebook, and it is invisible in the output.

</details>

---

### A4 · Put them together  ★★☆☆☆

Assemble the full table: the three features from the lecture plus the target, with incomplete rows dropped. Call it `tbl`.

In [ ]:
tbl = ...
tbl

<details>
<summary>💡 Hint</summary>

`pd.DataFrame({'name': series, ...})` builds the table, then `.dropna()` removes the rows where a window did not fit.

</details>

<details>
<summary>✅ Solution</summary>

```python
tbl = pd.DataFrame({
    'vol_20d': rets['AAPL'].rolling(20).std(),
    'vol_60d': rets['AAPL'].rolling(60).std(),
    'ret_20d': rets['AAPL'].rolling(20).mean(),
})
tbl['vol_next'] = rets['AAPL'].rolling(20).std().shift(-20)
tbl = tbl.dropna()
tbl
```

2,436 rows and 4 columns. The rows lost at the start are where the sixty-day window had not filled, and the twenty at the end are where the target reaches past the last date in the file.

</details>

---

### A5 · Split by date  ★★☆☆☆  · revisits S3

Cut `table` into `tr` (everything up to the end of 2022) and `te` (2023 onwards), and print how many rows each has.

In [ ]:
tr = ...
te = ...
print('train:', ...)
print('test :', ...)

<details>
<summary>💡 Hint</summary>

`frame.loc[:'2022-12-31']` and `frame.loc['2023-01-01':]`.

</details>

<details>
<summary>✅ Solution</summary>

```python
tr = table.loc[:'2022-12-31']
te = table.loc['2023-01-01':]
print('train:', len(tr))
print('test :', len(te))
```

1,954 training rows and 482 test rows, so the test block is about 20% of the sample and it is the most recent part of it.

</details>

---

### A6 · Prove the split does not overlap  ★★★☆☆  · revisits S3

Print the **last date in the training block** and the **first date in the test block**. They should not be the same day.

In [ ]:
last_train = ...
first_test = ...
print('last train row:', ...)
print('first test row:', ...)

<details>
<summary>💡 Hint 1</summary>

`frame.index` holds the dates, and `[-1]` and `[0]` pick the ends.

</details>

<details>
<summary>💡 Hint 2</summary>

`train.index[-1]` and `test.index[0]`. Add `.date()` if you want to lose the time part.

</details>

<details>
<summary>✅ Solution</summary>

```python
last_train = train.index[-1]
first_test = test.index[0]
print('last train row:', last_train.date())
print('first test row:', first_test.date())
```

2022-12-30 then 2023-01-03. Worth checking once: an off-by-one here puts a training day inside the test block, and nothing will warn you.

</details>

---

## 🔧 B · Your first fit

Three lines to fit a model, and rather more than three to understand what came out.

### B1 · X and y  ★☆☆☆☆

Build `X_train` from the single column `vol_20d`, and `y_train` from `vol_next`, both out of `train`. Print the shape of each.

In [ ]:
X_train = ...
y_train = ...
print('X:', ...)
print('y:', ...)

<details>
<summary>💡 Hint</summary>

`X` needs a **list** of column names inside the brackets, so it stays a table. `y` is one column, so one pair of brackets.

</details>

<details>
<summary>✅ Solution</summary>

```python
X_train = train[['vol_20d']]
y_train = train['vol_next']
print('X:', X_train.shape)
print('y:', y_train.shape)
```

`X` is `(1954, 1)` and `y` is `(1954,)`. Two dimensions against one, which is the difference the double brackets make.

</details>

---

### B2 · One bracket or two  ★★☆☆☆

Print the shape of `train['vol_20d']` and of `train[['vol_20d']]` side by side, so the difference is on the screen rather than in your memory.

In [ ]:
one_pair = ...
two_pairs = ...
print('one pair :', ...)
print('two pairs:', ...)

<details>
<summary>💡 Hint</summary>

Both are just selections out of `train`. Ask each one for its `.shape`.

</details>

<details>
<summary>✅ Solution</summary>

```python
one_pair = train['vol_20d']
two_pairs = train[['vol_20d']]
print('one pair :', one_pair.shape)
print('two pairs:', two_pairs.shape)
```

`(1954,)` against `(1954, 1)`. The first is a Series and scikit-learn refuses it as `X`. The second is a table with one column, which is what `X` always has to be.

</details>

---

### B3 · Fit it  ★★☆☆☆

Fit a `LinearRegression` on `X_train` and `y_train`. The model object is already made for you.

In [ ]:
X_train = train[['vol_20d']]
y_train = train['vol_next']

model = LinearRegression()

...        # fit the model here

model

<details>
<summary>💡 Hint</summary>

`model.fit(X, y)`. Features first, target second.

</details>

<details>
<summary>✅ Solution</summary>

```python
X_train = train[['vol_20d']]
y_train = train['vol_next']

model = LinearRegression()
model.fit(X_train, y_train)

model
```

`.fit()` prints nothing and returns the model. What changed is stored on the object, which is what B4 goes looking for.

</details>

---

### B4 · What it learned  ★☆☆☆☆

Print the intercept and the coefficients of the fitted model.

In [ ]:
model = LinearRegression()
model.fit(train[['vol_20d']], train['vol_next'])

print('intercept:', ...)
print('coefficients:', ...)

<details>
<summary>💡 Hint</summary>

Both are attributes with a trailing underscore, which marks them as computed from the data.

</details>

<details>
<summary>✅ Solution</summary>

```python
model = LinearRegression()
model.fit(train[['vol_20d']], train['vol_next'])

print('intercept:', model.intercept_)
print('coefficients:', model.coef_)
```

An intercept of about 0.8812 and one coefficient of about 0.4865. `coef_` is an array even when there is only one feature, because there usually is more than one.

</details>

---

### B5 · Write the fitted rule out  ★★★☆☆  · revisits S1

Print the fitted model as an equation, rounded to two decimals, using an f-string. Aim for something like `vol_next = 0.88 + 0.49 * vol_20d`.

In [ ]:
model = LinearRegression()
model.fit(train[['vol_20d']], train['vol_next'])

equation = ...
print(equation)

<details>
<summary>💡 Hint 1</summary>

`model.coef_[0]` is the single coefficient. `f'{value:.2f}'` rounds inside an f-string.

</details>

<details>
<summary>💡 Hint 2</summary>

f"vol_next = {model.intercept_:.2f} + {model.coef_[0]:.2f} * vol_20d"

</details>

<details>
<summary>✅ Solution</summary>

```python
model = LinearRegression()
model.fit(train[['vol_20d']], train['vol_next'])

equation = f'vol_next = {model.intercept_:.2f} + {model.coef_[0]:.2f} * vol_20d'
print(equation)
```

`vol_next = 0.88 + 0.49 * vol_20d`. A slope near a half rather than near one is mean reversion: only about half of this month's unusual volatility carries into next month.

</details>

---

### B6 · Predict  ★★☆☆☆

Predict on the test block and show the first five predictions.

In [ ]:
model = LinearRegression()
model.fit(train[['vol_20d']], train['vol_next'])

predictions = ...
predictions

<details>
<summary>💡 Hint</summary>

`model.predict(X)` wants the same columns, in the same shape, out of `test`.

</details>

<details>
<summary>✅ Solution</summary>

```python
model = LinearRegression()
model.fit(train[['vol_20d']], train['vol_next'])

predictions = model.predict(test[['vol_20d']])
predictions[:5]
```

A plain numpy array, one number per test row, in the same order as the rows. No dates attached, which is why the next exercise puts them back.

</details>

---

### B7 · Actual, predicted, residual  ★★★☆☆  · revisits S3

Build a DataFrame called `check` with three columns, `actual`, `predicted` and `residual`, for the **first five** test rows, indexed by their dates.

In [ ]:
model = LinearRegression()
model.fit(train[['vol_20d']], train['vol_next'])
predictions = model.predict(test[['vol_20d']])

check = ...
check

<details>
<summary>💡 Hint 1</summary>

`test['vol_next'].values[:5]` and `predictions[:5]` are the two number columns, and the residual is the first minus the second.

</details>

<details>
<summary>💡 Hint 2</summary>

Pass `index=test.index[:5]` to `pd.DataFrame` so the dates come back.

</details>

<details>
<summary>✅ Solution</summary>

```python
model = LinearRegression()
model.fit(train[['vol_20d']], train['vol_next'])
predictions = model.predict(test[['vol_20d']])

check = pd.DataFrame({
    'actual': test['vol_next'].values[:5],
    'predicted': predictions[:5],
    'residual': test['vol_next'].values[:5] - predictions[:5],
}, index=test.index[:5])
check
```

The residual column is the one Session 4 called $e_i = y_i - \hat{y}_i$. Reading five of them tells you more about a model than any single score does.

</details>

---

## 📏 C · Scoring, and something to beat

A score on its own says nothing. These exercises build the two rules any model has to beat before it is worth reporting.

### C1 · Mean squared error  ★☆☆☆☆

Compute the mean squared error of the model's predictions on the test block.

In [ ]:
model = LinearRegression()
model.fit(train[['vol_20d']], train['vol_next'])
predictions = model.predict(test[['vol_20d']])

mse = ...
print(mse)

<details>
<summary>💡 Hint</summary>

`mean_squared_error(y_true, y_pred)`, with the true values first.

</details>

<details>
<summary>✅ Solution</summary>

```python
model = LinearRegression()
model.fit(train[['vol_20d']], train['vol_next'])
predictions = model.predict(test[['vol_20d']])

mse = mean_squared_error(test['vol_next'], predictions)
print(mse)
```

About 0.1645, in squared percentage points. Nobody can picture that unit, which is what C2 fixes.

</details>

---

### C2 · Root mean squared error  ★☆☆☆☆

Take the square root, so the number is back in percentage points.

In [ ]:
model = LinearRegression()
model.fit(train[['vol_20d']], train['vol_next'])
predictions = model.predict(test[['vol_20d']])

rmse = ...
print(rmse)

<details>
<summary>💡 Hint</summary>

`np.sqrt` of the mean squared error.

</details>

<details>
<summary>✅ Solution</summary>

```python
model = LinearRegression()
model.fit(train[['vol_20d']], train['vol_next'])
predictions = model.predict(test[['vol_20d']])

rmse = np.sqrt(mean_squared_error(test['vol_next'], predictions))
print(rmse)
```

About 0.4056. The forecast is out by roughly 0.41 percentage points on a typical day.

</details>

---

### C3 · Baseline one: guess the average  ★★☆☆☆

Score the rule that ignores the features and predicts the **training** average on every test day.

In [ ]:
guess = ...
flat = ...

base_rmse = ...
print(base_rmse)

<details>
<summary>💡 Hint 1</summary>

The guess is `train['vol_next'].mean()`. Careful: the training mean, not the test mean.

</details>

<details>
<summary>💡 Hint 2</summary>

`np.full(len(test), guess)` repeats it once per test row.

</details>

<details>
<summary>✅ Solution</summary>

```python
guess = train['vol_next'].mean()
flat = np.full(len(test), guess)

base_rmse = np.sqrt(mean_squared_error(test['vol_next'], flat))
print(base_rmse)
```

0.5197. Using the test average instead would need information nobody had at the time, and it would make this baseline look better than it is.

</details>

---

### C4 · Baseline two: repeat this month  ★★☆☆☆

Score the rule that predicts next month's volatility to be whatever the last twenty days delivered. No fitting at all.

In [ ]:
pers_rmse = ...
print(pers_rmse)

<details>
<summary>💡 Hint</summary>

The prediction column is already sitting in the test block: `test['vol_20d']`.

</details>

<details>
<summary>✅ Solution</summary>

```python
pers_rmse = np.sqrt(mean_squared_error(test['vol_next'], test['vol_20d']))
print(pers_rmse)
```

0.4173, which already beats the average by a distance. Volatility clusters, so repeating the recent past is a genuinely strong rule and the one a model really has to beat.

</details>

---

### C5 · All three, in a dictionary  ★★★☆☆  · revisits S2

Collect the three scores into a dictionary called `scores`, then print them from best to worst.

In [ ]:
model = LinearRegression()
model.fit(train[['vol_20d']], train['vol_next'])

scores = {
    'model': ...,
    'average': ...,
    'persistence': ...,
}

# print them smallest first
for name in scores:
    ...

<details>
<summary>💡 Hint 1</summary>

Each value is the same `np.sqrt(mean_squared_error(...))` you have written three times already.

</details>

<details>
<summary>💡 Hint 2</summary>

`sorted(scores, key=scores.get)` gives the keys in order of their values, smallest first.

</details>

<details>
<summary>✅ Solution</summary>

```python
model = LinearRegression()
model.fit(train[['vol_20d']], train['vol_next'])

scores = {
    'model': np.sqrt(mean_squared_error(test['vol_next'], model.predict(test[['vol_20d']]))),
    'average': np.sqrt(mean_squared_error(test['vol_next'], np.full(len(test), train['vol_next'].mean()))),
    'persistence': np.sqrt(mean_squared_error(test['vol_next'], test['vol_20d'])),
}

for name in sorted(scores, key=scores.get):
    print(f'{name:12} {scores[name]:.4f}')
```

model 0.4056, persistence 0.4173, average 0.5197. The model wins by about 3% over persistence, which is a normal margin when forecasting financial data.

</details>

---

### C6 · Out-of-sample R squared  ★★★☆☆  · revisits S4

Compute how much better the model is than the average, as an $R^2$. The baseline in the denominator must be the **training** mean.

$$R^2 = 1 - \frac{\sum_i (y_i-\hat{y}_i)^2}{\sum_i (y_i-\bar{y}_{\text{train}})^2}$$

In [ ]:
model = LinearRegression()
model.fit(train[['vol_20d']], train['vol_next'])
predictions = model.predict(test[['vol_20d']])

rss = ...
tss = ...
r2 = ...
print(r2)

<details>
<summary>💡 Hint 1</summary>

`rss` is `((test['vol_next'] - predictions) ** 2).sum()`.

</details>

<details>
<summary>💡 Hint 2</summary>

`tss` is the same thing with `train['vol_next'].mean()` in place of the predictions.

</details>

<details>
<summary>✅ Solution</summary>

```python
model = LinearRegression()
model.fit(train[['vol_20d']], train['vol_next'])
predictions = model.predict(test[['vol_20d']])

rss = ((test['vol_next'] - predictions) ** 2).sum()
tss = ((test['vol_next'] - train['vol_next'].mean()) ** 2).sum()
r2 = 1 - rss / tss
print(r2)
```

About 0.391. On days it had never seen, the model accounts for roughly 39% of the variation that the training average leaves unexplained.

</details>

---

### C7 · A function that does all of it  ★★★★☆  · revisits S2

Write `test_rmse(columns)`: it fits a linear regression on those columns of `train`, predicts on `test`, and returns the RMSE. Then call it on `A`, on `B` and on `C`.

In [ ]:
def test_rmse(columns):
    ...

print('A:', ...)
print('B:', ...)
print('C:', ...)

<details>
<summary>💡 Hint 1</summary>

The body is four lines: make the model, fit it on `train[columns]`, predict on `test[columns]`, return the root mean squared error.

</details>

<details>
<summary>💡 Hint 2</summary>

Remember `return`. A function with no `return` hands back `None`.

</details>

<details>
<summary>✅ Solution</summary>

```python
def test_rmse(columns):
    model = LinearRegression()
    model.fit(train[columns], train['vol_next'])
    predictions = model.predict(test[columns])
    return np.sqrt(mean_squared_error(test['vol_next'], predictions))

print('A:', test_rmse(A))
print('B:', test_rmse(B))
print('C:', test_rmse(C))
```

0.4056, 0.4108, 0.4255. Wrapping it in a function is worth the two minutes: everything from here compares feature sets, and you now do it in one line each.

</details>

---

### C8 · The same number, two ways  ★★★☆☆  · revisits S3

`mean_squared_error` is only a name for some arithmetic you can do yourself. Compute the RMSE from the residuals with plain array arithmetic, and check it against the library.

In [ ]:
model = LinearRegression()
model.fit(train[['vol_20d']], train['vol_next'])
predictions = model.predict(test[['vol_20d']])

errors = ...
by_hand = ...
from_library = ...

print('by hand    :', ...)
print('the library:', ...)

<details>
<summary>💡 Hint 1</summary>

The residuals are `test['vol_next'] - predictions`. Subtracting a whole column from a whole array at once is the vectorised arithmetic from Session 3.

</details>

<details>
<summary>💡 Hint 2</summary>

Square them, take the mean, take the square root. No loop is needed anywhere: `np.sqrt((errors ** 2).mean())`.

</details>

<details>
<summary>✅ Solution</summary>

```python
model = LinearRegression()
model.fit(train[['vol_20d']], train['vol_next'])
predictions = model.predict(test[['vol_20d']])

errors = test['vol_next'] - predictions
by_hand = np.sqrt((errors ** 2).mean())
from_library = np.sqrt(mean_squared_error(test['vol_next'], predictions))

print('by hand    :', by_hand)
print('the library:', from_library)
```

Both give 0.4056104809. Not close, identical, because they are the same four operations.

Doing it twice and checking the two agree is a habit worth keeping. It is how you find out that you passed the arguments to a metric the wrong way round, which is silent and produces a plausible number.

</details>

---

### C9 · Which way is it wrong  ★★★☆☆  · revisits S3

An RMSE says how big the errors are and nothing about their direction. Using the same residuals, find the share of test days where the model predicted **too much** volatility, and the average error.

In [ ]:
model = LinearRegression()
model.fit(train[['vol_20d']], train['vol_next'])
predictions = model.predict(test[['vol_20d']])
errors = test['vol_next'] - predictions

share_over = ...
mean_error = ...

print('over-predicted on:', ...)
print('mean error       :', ...)

<details>
<summary>💡 Hint 1</summary>

An error is `actual - predicted`, so the model predicted too much wherever the error is **negative**.

</details>

<details>
<summary>💡 Hint 2</summary>

`(errors < 0)` is a column of True and False. Its `.mean()` is the share that are True, exactly as in Session 3.

</details>

<details>
<summary>✅ Solution</summary>

```python
model = LinearRegression()
model.fit(train[['vol_20d']], train['vol_next'])
predictions = model.predict(test[['vol_20d']])
errors = test['vol_next'] - predictions

share_over = (errors < 0).mean()
mean_error = errors.mean()

print('over-predicted on:', round(share_over, 3))
print('mean error       :', round(mean_error, 5))
```

The model forecast too much volatility on **77% of test days**, and the average error is -0.23336 rather than zero.

That is a systematic bias, not bad luck. The training years contain 2020 and the test years do not, so a model fitted on the louder period expects more noise than 2023 and 2024 delivered. An RMSE on its own would never have shown you this, which is why a residual is worth looking at directly.

</details>

---

## 🗳️ D · Three candidates

Same model, same rows, same target. Only the columns change.

### D1 · Name them  ★☆☆☆☆

The setup cell defined `A`, `B` and `C`. Print each one with how many features it holds.

In [ ]:
for name, columns in [('A', A), ('B', B), ('C', C)]:
    ...

<details>
<summary>💡 Hint</summary>

Inside the loop, print the name, `len(columns)`, and the list itself.

</details>

<details>
<summary>✅ Solution</summary>

```python
for name, columns in [('A', A), ('B', B), ('C', C)]:
    print(name, len(columns), 'features:', columns)
```

One, two and three features. Everything in this section and the next two is about deciding between them.

</details>

---

### D2 · Score them on the rows they were fitted on  ★★☆☆☆  · revisits S2

Loop over the three candidates and print the **training** RMSE of each.

In [ ]:
for columns in [A, B, C]:
    model = LinearRegression()
    ...
    error = ...
    print(len(columns), 'features:', ...)

<details>
<summary>💡 Hint</summary>

Fit on `train[columns]` and score the predictions against `train['vol_next']`, on the same rows.

</details>

<details>
<summary>✅ Solution</summary>

```python
for columns in [A, B, C]:
    model = LinearRegression()
    model.fit(train[columns], train['vol_next'])
    error = np.sqrt(mean_squared_error(train['vol_next'], model.predict(train[columns])))
    print(len(columns), 'features:', round(error, 4))
```

0.7218, 0.7213, 0.7102. The error falls every time a column is added, which is what training error always does.

</details>

---

### D3 · Score them on rows they have never seen  ★★☆☆☆

Now the same loop, scored on the test block instead.

In [ ]:
for columns in [A, B, C]:
    model = LinearRegression()
    ...
    error = ...
    print(len(columns), 'features:', ...)

<details>
<summary>💡 Hint</summary>

Only the two places that say `train` in the scoring line change to `test`. The fit stays on the training rows.

</details>

<details>
<summary>✅ Solution</summary>

```python
for columns in [A, B, C]:
    model = LinearRegression()
    model.fit(train[columns], train['vol_next'])
    error = np.sqrt(mean_squared_error(test['vol_next'], model.predict(test[columns])))
    print(len(columns), 'features:', round(error, 4))
```

0.4056, 0.4108, 0.4255. The ranking reverses. The two extra columns were fitting noise rather than signal.

</details>

---

### D4 · The two winners, side by side  ★★★☆☆

Build a DataFrame called `comparison` with one row per candidate and two columns, `train` and `test`.

In [ ]:
rows = []
for name, columns in [('A', A), ('B', B), ('C', C)]:
    ...

comparison = ...
comparison

<details>
<summary>💡 Hint 1</summary>

Inside the loop, append a dictionary: `rows.append({'set': name, 'train': ..., 'test': ...})`.

</details>

<details>
<summary>💡 Hint 2</summary>

`pd.DataFrame(rows).set_index('set')` turns the list of dictionaries into a table.

</details>

<details>
<summary>✅ Solution</summary>

```python
rows = []
for name, columns in [('A', A), ('B', B), ('C', C)]:
    model = LinearRegression()
    model.fit(train[columns], train['vol_next'])
    rows.append({
        'set': name,
        'train': np.sqrt(mean_squared_error(train['vol_next'], model.predict(train[columns]))),
        'test': np.sqrt(mean_squared_error(test['vol_next'], model.predict(test[columns]))),
    })

comparison = pd.DataFrame(rows).set_index('set')
comparison
```

The `train` column falls as you go down and the `test` column rises. Two columns, two opposite orderings, and only one of them was available while the choice was being made.

</details>

---

### D5 · A column with nothing in it  ★★★★☆

Add a column of pure random numbers to the training block and fit `A` plus that column. Does the **training** error go up, down, or stay the same?

In [ ]:
noisy = train.copy()
noisy['noise'] = rng.normal(0, 1, len(train))

plain_error = ...
noisy_error = ...

print('vol_20d alone      :', ...)
print('vol_20d plus noise :', ...)

<details>
<summary>💡 Hint 1</summary>

Fit twice: once on `train[A]`, once on `noisy[['vol_20d', 'noise']]`. Score both on the training rows.

</details>

<details>
<summary>💡 Hint 2</summary>

The noise column cannot possibly help on new data. Watch what it does to the fit on these rows anyway.

</details>

<details>
<summary>✅ Solution</summary>

```python
noisy = train.copy()
noisy['noise'] = rng.normal(0, 1, len(train))

m1 = LinearRegression().fit(train[A], train['vol_next'])
m2 = LinearRegression().fit(noisy[['vol_20d', 'noise']], noisy['vol_next'])

plain_error = np.sqrt(mean_squared_error(train['vol_next'], m1.predict(train[A])))
noisy_error = np.sqrt(mean_squared_error(noisy['vol_next'], m2.predict(noisy[['vol_20d', 'noise']])))

print('vol_20d alone      :', round(plain_error, 8))
print('vol_20d plus noise :', round(noisy_error, 8))
```

0.72176292 against 0.72176154. The difference is tiny, and the direction is the whole point: adding a column of pure noise made the training fit **better**, never worse. It can only ever go that way, which is why training error cannot choose a model.

Eight decimals are needed to see it here because one random column against 1,954 rows has very little room. Give a model thirty noise columns and the same effect becomes large enough to fool you.

</details>

---

### D6 · Draw it  ★★★★☆  · revisits S3

Plot training RMSE and test RMSE against the number of features, both on one axes, with a legend.

In [ ]:
n_features = []
train_scores = []
test_scores = []

for columns in [A, B, C]:
    ...

fig, ax = plt.subplots(figsize=(7, 3.5))
...
plt.show()

<details>
<summary>💡 Hint 1</summary>

Fill the three lists inside the loop, then draw two `ax.plot(...)` lines with `marker='o'` and a `label=`.

</details>

<details>
<summary>💡 Hint 2</summary>

`ax.set_xticks([1, 2, 3])` keeps the axis honest, and `ax.legend()` shows the labels.

</details>

<details>
<summary>✅ Solution</summary>

```python
n_features = []
train_scores = []
test_scores = []

for columns in [A, B, C]:
    model = LinearRegression()
    model.fit(train[columns], train['vol_next'])
    n_features.append(len(columns))
    train_scores.append(np.sqrt(mean_squared_error(train['vol_next'], model.predict(train[columns]))))
    test_scores.append(np.sqrt(mean_squared_error(test['vol_next'], model.predict(test[columns]))))

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.plot(n_features, train_scores, marker='o', label='training')
ax.plot(n_features, test_scores, marker='o', label='test')
ax.set_xticks([1, 2, 3])
ax.set_xlabel('number of features')
ax.set_ylabel('RMSE (percentage points)')
ax.set_title('Training error falls. Test error does not.', loc='left')
ax.legend()
plt.show()
```

Two lines going opposite ways. This is the picture from Session 4, drawn from a model you fitted yourself rather than from an invented example.

</details>

---

## ✂️ E · One split is one experiment

The test rows have now been used to compare three models, which is exactly what they were not for. These exercises build the block that does the choosing.

### E1 · Cut the training block in two  ★★☆☆☆

Split `train` again by date: `fit_rows` up to the end of 2020, `val_rows` from 2021 onwards. Print the size of each.

In [ ]:
fit_rows = ...
val_rows = ...
print('fit       :', ...)
print('validation:', ...)

<details>
<summary>💡 Hint</summary>

Same `.loc` slicing as the first split, applied to `train` instead of `table`.

</details>

<details>
<summary>✅ Solution</summary>

```python
fit_rows = train.loc[:'2020-12-31']
val_rows = train.loc['2021-01-01':]
print('fit       :', len(fit_rows))
print('validation:', len(val_rows))
```

1,451 rows to fit on and 503 to compare on. The test block is not mentioned anywhere in this exercise, which is the point of it.

</details>

---

### E2 · Choose on the validation block  ★★☆☆☆

Score the three candidates on `val_rows` after fitting them on `fit_rows`, and say which wins.

In [ ]:
fit_rows = train.loc[:'2020-12-31']
val_rows = train.loc['2021-01-01':]

for columns in [A, B, C]:
    ...
    print(len(columns), 'features:', ...)

<details>
<summary>💡 Hint</summary>

The same loop as D3, with `fit_rows` where `train` was and `val_rows` where `test` was.

</details>

<details>
<summary>✅ Solution</summary>

```python
fit_rows = train.loc[:'2020-12-31']
val_rows = train.loc['2021-01-01':]

for columns in [A, B, C]:
    model = LinearRegression()
    model.fit(fit_rows[columns], fit_rows['vol_next'])
    error = np.sqrt(mean_squared_error(val_rows['vol_next'], model.predict(val_rows[columns])))
    print(len(columns), 'features:', round(error, 4))
```

0.5138, 0.5155, 0.5039. Three features win, and no test row was involved in finding that out.

</details>

---

### E3 · Move the cut one year  ★★★☆☆

Run the same comparison with the cut at the end of **2019** instead. Does the same candidate win?

In [ ]:
fit_rows = ...
val_rows = ...

for columns in [A, B, C]:
    ...
    print(len(columns), 'features:', ...)

<details>
<summary>💡 Hint</summary>

Only the two dates change: `train.loc[:'2019-12-31']` and `train.loc['2020-01-01':]`.

</details>

<details>
<summary>✅ Solution</summary>

```python
fit_rows = train.loc[:'2019-12-31']
val_rows = train.loc['2020-01-01':]

for columns in [A, B, C]:
    model = LinearRegression()
    model.fit(fit_rows[columns], fit_rows['vol_next'])
    error = np.sqrt(mean_squared_error(val_rows['vol_next'], model.predict(val_rows[columns])))
    print(len(columns), 'features:', round(error, 4))
```

0.9978, 1.0025, 1.0704. One feature wins now, and by a clear margin. The candidates did not change and the training block did not change. Only the cut date moved.

</details>

---

### E4 · Four cut dates, in a loop  ★★★★☆  · revisits S2

Loop over the cuts `'2018-12-31'`, `'2019-12-31'`, `'2020-12-31'` and `'2021-12-31'`, and record which candidate wins at each one in a dictionary called `winner`.

In [ ]:
winner = {}

for cut in ['2018-12-31', '2019-12-31', '2020-12-31', '2021-12-31']:
    ...

winner

<details>
<summary>💡 Hint 1</summary>

Inside the loop, build a small dictionary of the three scores first, then pick its smallest key.

</details>

<details>
<summary>💡 Hint 2</summary>

`min(scores, key=scores.get)` returns the key whose value is smallest.

</details>

<details>
<summary>✅ Solution</summary>

```python
winner = {}

for cut in ['2018-12-31', '2019-12-31', '2020-12-31', '2021-12-31']:
    fit_rows = train.loc[:cut]
    val_rows = train.loc[cut:]

    scores = {}
    for name, columns in [('A', A), ('B', B), ('C', C)]:
        model = LinearRegression()
        model.fit(fit_rows[columns], fit_rows['vol_next'])
        scores[name] = np.sqrt(mean_squared_error(val_rows['vol_next'], model.predict(val_rows[columns])))

    winner[cut] = min(scores, key=scores.get)

winner
```

Two of the cuts choose A and two choose C. A single validation block is a single experiment, and running one experiment is not enough to separate candidates this close together.

</details>

---

### E5 · How far does one number move  ★★★★★

For candidate `A` alone, find the **largest and smallest** validation RMSE across those four cut dates, and print the gap between them.

In [ ]:
a_scores = ...

print('lowest :', ...)
print('highest:', ...)
print('gap    :', ...)

<details>
<summary>💡 Hint 1</summary>

Collect the four numbers into a list first, with the same loop as E4 but only candidate `A`.

</details>

<details>
<summary>💡 Hint 2</summary>

`min(a_scores)`, `max(a_scores)`, and the difference between them.

</details>

<details>
<summary>✅ Solution</summary>

```python
a_scores = []

for cut in ['2018-12-31', '2019-12-31', '2020-12-31', '2021-12-31']:
    fit_rows = train.loc[:cut]
    val_rows = train.loc[cut:]
    model = LinearRegression()
    model.fit(fit_rows[A], fit_rows['vol_next'])
    a_scores.append(np.sqrt(mean_squared_error(val_rows['vol_next'], model.predict(val_rows[A]))))

print('lowest :', round(min(a_scores), 4))
print('highest:', round(max(a_scores), 4))
print('gap    :', round(max(a_scores) - min(a_scores), 4))
```

The same model, on the same training block, scores anywhere from 0.5141 to 0.9971 depending on where the cut goes. The gap between the candidates is far smaller than that, which is why one cut cannot decide between them.

</details>

---

## 🔄 F · Cross-validation

Several cut dates instead of one, and the average of what they say.

### F1 · Five folds  ★☆☆☆☆

Make a `TimeSeriesSplit` with five folds and print how many rows each fold fits on and scores on.

In [ ]:
folds = ...

# then loop over folds.split(train) and print the two sizes
...

<details>
<summary>💡 Hint</summary>

`folds.split(train)` yields a pair of row-position arrays per fold.

</details>

<details>
<summary>✅ Solution</summary>

```python
folds = TimeSeriesSplit(n_splits=5)

for fit_rows, score_rows in folds.split(train):
    print(len(fit_rows), 'fitted', len(score_rows), 'scored')
```

The fitted block grows by one step each time and the scored block stays the same size. Every scored block comes after the rows it was fitted on.

</details>

---

### F2 · Cross-validate candidate A  ★★☆☆☆

Run `cross_val_score` on candidate `A` with those folds, and print what comes back.

In [ ]:
folds = TimeSeriesSplit(n_splits=5)

scores = ...
print(scores)

<details>
<summary>💡 Hint</summary>

`cross_val_score(LinearRegression(), X, y, cv=folds, scoring='neg_root_mean_squared_error')`.

</details>

<details>
<summary>✅ Solution</summary>

```python
folds = TimeSeriesSplit(n_splits=5)

scores = cross_val_score(LinearRegression(), train[A], train['vol_next'],
                         cv=folds, scoring='neg_root_mean_squared_error')
print(scores)
```

Five numbers, all negative. scikit-learn reports every score so that larger is better, so it flips the sign of an error.

</details>

---

### F3 · Turn it into an error  ★☆☆☆☆

Flip the sign and take the average.

In [ ]:
folds = TimeSeriesSplit(n_splits=5)
scores = cross_val_score(LinearRegression(), train[A], train['vol_next'],
                         cv=folds, scoring='neg_root_mean_squared_error')

cv_error = ...
print(cv_error)

<details>
<summary>💡 Hint</summary>

`-scores.mean()`, or `(-scores).mean()`. Both work.

</details>

<details>
<summary>✅ Solution</summary>

```python
folds = TimeSeriesSplit(n_splits=5)
scores = cross_val_score(LinearRegression(), train[A], train['vol_next'],
                         cv=folds, scoring='neg_root_mean_squared_error')

cv_error = -scores.mean()
print(cv_error)
```

0.7404. Larger than the test RMSE of 0.4056, because every fold fits on less data than the final model will and scores on a harder stretch of history.

</details>

---

### F4 · The spread, not just the average  ★★☆☆☆

Print the five fold errors, their average, and their standard deviation.

In [ ]:
folds = TimeSeriesSplit(n_splits=5)
scores = cross_val_score(LinearRegression(), train[A], train['vol_next'],
                         cv=folds, scoring='neg_root_mean_squared_error')

fold_errors = ...
print('folds  :', ...)
print('average:', ...)
print('spread :', ...)

<details>
<summary>💡 Hint</summary>

`-scores` is the array of fold errors. It has `.mean()` and `.std()` like any numpy array.

</details>

<details>
<summary>✅ Solution</summary>

```python
folds = TimeSeriesSplit(n_splits=5)
scores = cross_val_score(LinearRegression(), train[A], train['vol_next'],
                         cv=folds, scoring='neg_root_mean_squared_error')

fold_errors = -scores
print('folds  :', fold_errors.round(3))
print('average:', round(fold_errors.mean(), 4))
print('spread :', round(fold_errors.std(), 4))
```

Four of the folds sit close together and one is roughly twice as large. An average that hides a fold like that is worth less than the two numbers together.

</details>

---

### F5 · All three candidates  ★★★☆☆  · revisits S2

Cross-validate `A`, `B` and `C` in a loop and print the average error of each.

In [ ]:
folds = TimeSeriesSplit(n_splits=5)

for columns in [A, B, C]:
    ...
    print(len(columns), 'features:', ...)

<details>
<summary>💡 Hint</summary>

One `cross_val_score` call per candidate, then `-scores.mean()`.

</details>

<details>
<summary>✅ Solution</summary>

```python
folds = TimeSeriesSplit(n_splits=5)

for columns in [A, B, C]:
    scores = cross_val_score(LinearRegression(), train[columns], train['vol_next'],
                             cv=folds, scoring='neg_root_mean_squared_error')
    print(len(columns), 'features:', round(-scores.mean(), 4))
```

0.7404, 0.7724, 0.7470. One feature wins, and the answer is now an average over five cut dates rather than whatever one arbitrary cut happened to say.

</details>

---

### F6 · Which period is the bad fold  ★★★★☆

Find the worst fold for candidate `A`, and print the first and last date it was scored on.

In [ ]:
folds = TimeSeriesSplit(n_splits=5)
scores = cross_val_score(LinearRegression(), train[A], train['vol_next'],
                         cv=folds, scoring='neg_root_mean_squared_error')
fold_errors = -scores

worst = ...
blocks = ...

print('worst fold:', ...)
print('scored from', ..., 'to', ...)

<details>
<summary>💡 Hint 1</summary>

`fold_errors.argmax()` gives the position of the largest error.

</details>

<details>
<summary>💡 Hint 2</summary>

`blocks = list(folds.split(train))` lets you index the folds. Each entry is a pair, and the second half holds the scored row positions.

</details>

<details>
<summary>✅ Solution</summary>

```python
folds = TimeSeriesSplit(n_splits=5)
scores = cross_val_score(LinearRegression(), train[A], train['vol_next'],
                         cv=folds, scoring='neg_root_mean_squared_error')
fold_errors = -scores

worst = fold_errors.argmax()
blocks = list(folds.split(train))
scored_rows = blocks[worst][1]

print('worst fold:', worst + 1)
print('scored from', train.index[scored_rows[0]].date(), 'to', train.index[scored_rows[-1]].date())
```

Fold 3, scored from 2019-02-20 to 2020-06-03. It contains March 2020. A model fitted on the quiet years before it had never seen a month like that, and the fold score says so.

</details>

---

### F7 · Ten folds instead of five  ★★☆☆☆

Run the same comparison with `n_splits=10` and see whether the winner holds.

In [ ]:
folds = ...

for columns in [A, B, C]:
    ...
    print(len(columns), 'features:', ...)

<details>
<summary>💡 Hint</summary>

Only the `n_splits` argument changes.

</details>

<details>
<summary>✅ Solution</summary>

```python
folds = TimeSeriesSplit(n_splits=10)

for columns in [A, B, C]:
    scores = cross_val_score(LinearRegression(), train[columns], train['vol_next'],
                             cv=folds, scoring='neg_root_mean_squared_error')
    print(len(columns), 'features:', round(-scores.mean(), 4))
```

0.6918, 0.6997, 0.6900. Every number is lower, because each fold now fits on more rows and scores on a shorter stretch.

Note that the winner changed: with ten folds the three candidates sit within 0.01 of each other and the order flips. That is worth knowing rather than hiding. It says these three are not really distinguishable on this data, so the honest report is that the simplest one is as good as any, not that it is provably best.

</details>

---

### F8 · Draw the folds  ★★★★☆  · revisits S3

Draw the five fold errors for candidate `A` as a bar chart, with a horizontal line at their average.

In [ ]:
folds = TimeSeriesSplit(n_splits=5)
scores = cross_val_score(LinearRegression(), train[A], train['vol_next'],
                         cv=folds, scoring='neg_root_mean_squared_error')
fold_errors = -scores

fig, ax = plt.subplots(figsize=(7, 3))
...
plt.show()

<details>
<summary>💡 Hint 1</summary>

`ax.bar(range(1, 6), fold_errors)` draws the five bars.

</details>

<details>
<summary>💡 Hint 2</summary>

`ax.axhline(fold_errors.mean())` is the average line, and `label=` plus `ax.legend()` explains it.

</details>

<details>
<summary>✅ Solution</summary>

```python
folds = TimeSeriesSplit(n_splits=5)
scores = cross_val_score(LinearRegression(), train[A], train['vol_next'],
                         cv=folds, scoring='neg_root_mean_squared_error')
fold_errors = -scores

fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(range(1, 6), fold_errors)
ax.axhline(fold_errors.mean(), color='black', linestyle='--', label='average')
ax.set_xlabel('fold')
ax.set_ylabel('RMSE (percentage points)')
ax.set_title('One fold is twice as hard as the others', loc='left')
ax.legend()
plt.show()
```

One bar stands well above the line. A picture like this belongs next to any cross-validation number you report, because the average on its own cannot show it.

</details>

---

## 📆 G · Folds and the calendar

What the textbook version of cross-validation does to a table whose rows overlap.

### G1 · The textbook folds  ★★☆☆☆

Run the same three-candidate comparison with `KFold(n_splits=5, shuffle=True, random_state=0)`.

In [ ]:
folds = ...

for columns in [A, B, C]:
    ...
    print(len(columns), 'features:', ...)

<details>
<summary>💡 Hint</summary>

Only the splitter changes. Everything else is the loop from F5.

</details>

<details>
<summary>✅ Solution</summary>

```python
folds = KFold(n_splits=5, shuffle=True, random_state=0)

for columns in [A, B, C]:
    scores = cross_val_score(LinearRegression(), train[columns], train['vol_next'],
                             cv=folds, scoring='neg_root_mean_squared_error')
    print(len(columns), 'features:', round(-scores.mean(), 4))
```

0.7199, 0.7198, 0.7087. Three features win, which is the candidate the test rows called worst.

</details>

---

### G2 · Three verdicts on one table  ★★★☆☆

Put the winner from each of the three methods into a dictionary: shuffled `KFold`, `TimeSeriesSplit`, and the test rows.

In [ ]:
def best_of(cv_object):
    ...

verdicts = {
    'shuffled KFold': ...,
    'TimeSeriesSplit': ...,
    'the test rows': ...,
}
verdicts

<details>
<summary>💡 Hint 1</summary>

`best_of` can build a dictionary of the three averages and return `min(scores, key=scores.get)`.

</details>

<details>
<summary>💡 Hint 2</summary>

For the test rows there are no folds: fit on all of `train` and score on `test`, then take the smallest.

</details>

<details>
<summary>✅ Solution</summary>

```python
def best_of(cv_object):
    scores = {}
    for name, columns in [('A', A), ('B', B), ('C', C)]:
        s = cross_val_score(LinearRegression(), train[columns], train['vol_next'],
                            cv=cv_object, scoring='neg_root_mean_squared_error')
        scores[name] = -s.mean()
    return min(scores, key=scores.get)

test_scores = {}
for name, columns in [('A', A), ('B', B), ('C', C)]:
    model = LinearRegression().fit(train[columns], train['vol_next'])
    test_scores[name] = np.sqrt(mean_squared_error(test['vol_next'], model.predict(test[columns])))

verdicts = {
    'shuffled KFold': best_of(KFold(n_splits=5, shuffle=True, random_state=0)),
    'TimeSeriesSplit': best_of(TimeSeriesSplit(n_splits=5)),
    'the test rows': min(test_scores, key=test_scores.get),
}
verdicts
```

`TimeSeriesSplit` agrees with the test rows and shuffled `KFold` does not. On a table like this one, a shuffled score can leave you with the wrong model, not just the wrong number.

</details>

---

### G3 · Why the shuffle helps the model cheat  ★★★☆☆  · revisits S3

Show how similar two neighbouring rows are. Print the average size of the change in `vol_20d` from one row to the next, next to the standard deviation of the column itself.

In [ ]:
step = ...
spread = ...

print('average day-to-day change:', ...)
print('standard deviation   :', ...)

<details>
<summary>💡 Hint 1</summary>

`table['vol_20d'] - table['vol_20d'].shift(1)` is the change from one row to the next.

</details>

<details>
<summary>💡 Hint 2</summary>

Take `.abs().mean()` of that, and compare it with `table['vol_20d'].std()`.

</details>

<details>
<summary>✅ Solution</summary>

```python
step = (table['vol_20d'] - table['vol_20d'].shift(1)).abs().mean()
spread = table['vol_20d'].std()

print('average day-to-day change:', round(step, 4))
print('standard deviation   :', round(spread, 4))
```

A typical day-to-day change of 0.0645 against a spread of 0.7687, so consecutive rows differ by about 8% of the column's own variation. Two neighbouring rows share nineteen of their twenty days. Put one in the fitting block and the other in the scored block and the score comes out too good.

</details>

---

### G4 · Leave a gap  ★★☆☆☆

Cross-validate candidate `A` with a twenty-row gap between each fitted block and the block it is scored on.

In [ ]:
folds = ...

# then cross-validate candidate A with those folds and print the average
...

<details>
<summary>💡 Hint</summary>

`TimeSeriesSplit(n_splits=5, gap=20)`.

</details>

<details>
<summary>✅ Solution</summary>

```python
folds = TimeSeriesSplit(n_splits=5, gap=20)

scores = cross_val_score(LinearRegression(), train[A], train['vol_next'],
                         cv=folds, scoring='neg_root_mean_squared_error')
print(round(-scores.mean(), 4))
```

0.7394, against 0.7404 with no gap. A small change here, because twenty rows are little against fitting blocks of several hundred. The same correction matters far more when the target spans a longer window.

</details>

---

### G5 · Forget the old years  ★★★☆☆

Cross-validate candidate `A` with a **rolling** window of 500 rows instead of an expanding one, and compare.

In [ ]:
expanding = ...
rolling = ...

print('expanding:', ...)
print('rolling  :', ...)

<details>
<summary>💡 Hint</summary>

`max_train_size=500` turns an expanding window into a rolling one.

</details>

<details>
<summary>✅ Solution</summary>

```python
expanding = cross_val_score(LinearRegression(), train[A], train['vol_next'],
                            cv=TimeSeriesSplit(n_splits=5),
                            scoring='neg_root_mean_squared_error')
rolling = cross_val_score(LinearRegression(), train[A], train['vol_next'],
                          cv=TimeSeriesSplit(n_splits=5, max_train_size=500),
                          scoring='neg_root_mean_squared_error')

print('expanding:', round(-expanding.mean(), 4))
print('rolling  :', round(-rolling.mean(), 4))
```

0.7404 expanding against 0.7378 rolling. Very close here, which says that the older years are neither much help nor much harm. On data where the relationship really has moved, the two numbers separate.

</details>

---

### G6 · Rank the estimates  ★★★★☆

Build a dictionary of the cross-validation error for candidate `A` under four arrangements: shuffled `KFold`, expanding, expanding with a gap, and rolling. Print them from most optimistic to most pessimistic.

In [ ]:
arrangements = {
    'shuffled': ...,
    'expanding': ...,
    'expanding + gap': ...,
    'rolling': ...,
}

# print them most optimistic first
for name in arrangements:
    ...

<details>
<summary>💡 Hint 1</summary>

Each value is one `cross_val_score` call with a different `cv=` object, followed by `-scores.mean()`.

</details>

<details>
<summary>💡 Hint 2</summary>

`sorted(arrangements, key=arrangements.get)` puts the smallest, most optimistic one first.

</details>

<details>
<summary>✅ Solution</summary>

```python
def cv_error(cv_object):
    s = cross_val_score(LinearRegression(), train[A], train['vol_next'],
                        cv=cv_object, scoring='neg_root_mean_squared_error')
    return -s.mean()

arrangements = {
    'shuffled': cv_error(KFold(n_splits=5, shuffle=True, random_state=0)),
    'expanding': cv_error(TimeSeriesSplit(n_splits=5)),
    'expanding + gap': cv_error(TimeSeriesSplit(n_splits=5, gap=20)),
    'rolling': cv_error(TimeSeriesSplit(n_splits=5, max_train_size=500)),
}

for name in sorted(arrangements, key=arrangements.get):
    print(f'{name:16} {arrangements[name]:.4f}')
```

The shuffled arrangement is the most optimistic at 0.7199, and it is the only one of the four that lets a fold be scored by a model fitted on its neighbours. Every arrangement that respects the calendar gives a worse and more honest number.

</details>

---

## 🧮 H · Counting the cost of a feature

AIC and BIC score fit and size in one number, without any folds at all.

### H1 · Training MSE for each candidate  ★★☆☆☆

Both formulas start from the mean squared error on the rows the model was fitted on. Collect it for the three candidates in a dictionary called `train_mse`.

In [ ]:
train_mse = {}

for name, columns in [('A', A), ('B', B), ('C', C)]:
    ...

train_mse

<details>
<summary>💡 Hint</summary>

`mean_squared_error(train['vol_next'], model.predict(train[columns]))`, with no square root this time.

</details>

<details>
<summary>✅ Solution</summary>

```python
train_mse = {}

for name, columns in [('A', A), ('B', B), ('C', C)]:
    model = LinearRegression()
    model.fit(train[columns], train['vol_next'])
    train_mse[name] = round(float(mean_squared_error(train['vol_next'], model.predict(train[columns]))), 5)

train_mse
```

0.52094, 0.52027, 0.50433. Falling as columns are added, as it always does.

</details>

---

### H2 · AIC  ★★★☆☆

Compute the AIC of each candidate.

$$\text{AIC} = n\log(\text{MSE}) + 2d$$

Here $n$ is the number of training rows and $d$ is the number of parameters: one per feature, plus the intercept.

In [ ]:
n = ...
aic = {}

for name, columns in [('A', A), ('B', B), ('C', C)]:
    ...

aic

<details>
<summary>💡 Hint 1</summary>

`n` is `len(train)`, and `d` is `len(columns) + 1`.

</details>

<details>
<summary>💡 Hint 2</summary>

`np.log` is the natural logarithm. Fit the model, get its training MSE, then apply the formula.

</details>

<details>
<summary>✅ Solution</summary>

```python
n = len(train)
aic = {}

for name, columns in [('A', A), ('B', B), ('C', C)]:
    model = LinearRegression()
    model.fit(train[columns], train['vol_next'])
    mse = mean_squared_error(train['vol_next'], model.predict(train[columns]))
    d = len(columns) + 1
    aic[name] = round(float(n * np.log(mse) + 2 * d), 1)

aic
```

-1270, -1271, -1330. Smaller is better, so AIC prefers candidate C.

</details>

---

### H3 · BIC  ★★★☆☆

Now BIC, which charges more per parameter.

$$\text{BIC} = n\log(\text{MSE}) + d\log(n)$$

In [ ]:
n = len(train)
bic = {}

for name, columns in [('A', A), ('B', B), ('C', C)]:
    ...

bic

<details>
<summary>💡 Hint</summary>

The only change from H2 is `2 * d` becoming `d * np.log(n)`.

</details>

<details>
<summary>✅ Solution</summary>

```python
n = len(train)
bic = {}

for name, columns in [('A', A), ('B', B), ('C', C)]:
    model = LinearRegression()
    model.fit(train[columns], train['vol_next'])
    mse = mean_squared_error(train['vol_next'], model.predict(train[columns]))
    d = len(columns) + 1
    bic[name] = round(float(n * np.log(mse) + d * np.log(n)), 1)

bic
```

-1259, -1254, -1307. BIC prefers C. With 1,954 rows, $\log(n)$ is about 7.6, so BIC charges roughly 3.8 times as much per parameter as AIC does.

</details>

---

### H4 · Who agrees with whom  ★★★☆☆

Put the winner according to AIC, BIC, shuffled `KFold` and `TimeSeriesSplit` into one dictionary.

In [ ]:
picks = {
    'AIC': ...,
    'BIC': ...,
    'shuffled KFold': ...,
    'TimeSeriesSplit': ...,
}
picks

<details>
<summary>💡 Hint</summary>

You already wrote the AIC and BIC dictionaries in H2 and H3, and the two cross-validation loops in F5 and G1. `min(d, key=d.get)` picks the smallest from each.

</details>

<details>
<summary>✅ Solution</summary>

```python
n = len(train)
aic, bic, kf, ts = {}, {}, {}, {}

for name, columns in [('A', A), ('B', B), ('C', C)]:
    model = LinearRegression()
    model.fit(train[columns], train['vol_next'])
    mse = mean_squared_error(train['vol_next'], model.predict(train[columns]))
    d = len(columns) + 1
    aic[name] = n * np.log(mse) + 2 * d
    bic[name] = n * np.log(mse) + d * np.log(n)
    kf[name] = -cross_val_score(LinearRegression(), train[columns], train['vol_next'],
                                cv=KFold(n_splits=5, shuffle=True, random_state=0),
                                scoring='neg_root_mean_squared_error').mean()
    ts[name] = -cross_val_score(LinearRegression(), train[columns], train['vol_next'],
                                cv=TimeSeriesSplit(n_splits=5),
                                scoring='neg_root_mean_squared_error').mean()

picks = {
    'AIC': min(aic, key=aic.get),
    'BIC': min(bic, key=bic.get),
    'shuffled KFold': min(kf, key=kf.get),
    'TimeSeriesSplit': min(ts, key=ts.get),
}
picks
```

AIC, BIC and shuffled `KFold` all choose C. Only `TimeSeriesSplit` chooses A, and only `TimeSeriesSplit` agrees with the test rows. The three that disagree all treat the rows as independent pieces of evidence, and these rows are nothing of the kind.

</details>

---

### H5 · How many observations are really there  ★★★★★

Each row shares nineteen of its twenty days with the row before it, so 1,954 rows are nowhere near 1,954 independent observations. Recompute BIC using `len(train) // 20` in place of $n$, and see whether it changes its mind.

In [ ]:
n_eff = ...
bic_eff = {}

for name, columns in [('A', A), ('B', B), ('C', C)]:
    ...

print(bic_eff)
print('picks:', ...)

<details>
<summary>💡 Hint 1</summary>

`n_eff = len(train) // 20`. Use it in **both** places where $n$ appears in the formula.

</details>

<details>
<summary>💡 Hint 2</summary>

The MSE itself does not change. Only the two $n$ terms do.

</details>

<details>
<summary>✅ Solution</summary>

```python
n_eff = len(train) // 20
bic_eff = {}

for name, columns in [('A', A), ('B', B), ('C', C)]:
    model = LinearRegression()
    model.fit(train[columns], train['vol_next'])
    mse = mean_squared_error(train['vol_next'], model.predict(train[columns]))
    d = len(columns) + 1
    bic_eff[name] = round(float(n_eff * np.log(mse) + d * np.log(n_eff)), 1)

print(bic_eff)
print('picks:', min(bic_eff, key=bic_eff.get))
```

With $n = 97$ instead of 1,954, BIC picks A. The MSE never moved, so the whole change came from admitting how much evidence there really is.

That is the honest reading of H4: AIC and BIC are not wrong, they were handed an $n$ that was twenty times too large. Cross-validation with folds that respect the calendar never needs that number at all, which is why it survives on data like this.

</details>

---

## 🏗️ I · The whole workflow, on a stock the lecture never used

Nothing new here. The point is to run the six steps end to end without the lecture holding your hand.

### I1 · Build Microsoft's table  ★★☆☆☆  · revisits S4

Build the same four-column table for `MSFT`, drop the incomplete rows, and split it at the end of 2022 into `m_train` and `m_test`.

In [ ]:
msft = ...

m_train = ...
m_test = ...
print(m_train)
print(m_test)

<details>
<summary>💡 Hint</summary>

This is exercise A4 and A5 again with a different ticker.

</details>

<details>
<summary>✅ Solution</summary>

```python
msft = pd.DataFrame({
    'vol_20d': rets['MSFT'].rolling(20).std(),
    'vol_60d': rets['MSFT'].rolling(60).std(),
    'ret_20d': rets['MSFT'].rolling(20).mean(),
})
msft['vol_next'] = rets['MSFT'].rolling(20).std().shift(-20)
msft = msft.dropna()

m_train = msft.loc[:'2022-12-31']
m_test = msft.loc['2023-01-01':]
print(len(m_train), len(m_test))
```

The same shape as Apple's table, because the windows and the date range are the same.

</details>

---

### I2 · Choose a candidate, honestly  ★★☆☆☆

Cross-validate the three candidates on `m_train` with five time-ordered folds, and print each average.

In [ ]:
msft = pd.DataFrame({
    'vol_20d': rets['MSFT'].rolling(20).std(),
    'vol_60d': rets['MSFT'].rolling(60).std(),
    'ret_20d': rets['MSFT'].rolling(20).mean(),
})
msft['vol_next'] = rets['MSFT'].rolling(20).std().shift(-20)
msft = msft.dropna()
m_train = msft.loc[:'2022-12-31']

folds = TimeSeriesSplit(n_splits=5)

for columns in [A, B, C]:
    ...

<details>
<summary>💡 Hint</summary>

Exactly the loop from F5, with `m_train` in place of `train`.

</details>

<details>
<summary>✅ Solution</summary>

```python
msft = pd.DataFrame({
    'vol_20d': rets['MSFT'].rolling(20).std(),
    'vol_60d': rets['MSFT'].rolling(60).std(),
    'ret_20d': rets['MSFT'].rolling(20).mean(),
})
msft['vol_next'] = rets['MSFT'].rolling(20).std().shift(-20)
msft = msft.dropna()
m_train = msft.loc[:'2022-12-31']

folds = TimeSeriesSplit(n_splits=5)

for columns in [A, B, C]:
    scores = cross_val_score(LinearRegression(), m_train[columns], m_train['vol_next'],
                             cv=folds, scoring='neg_root_mean_squared_error')
    print(len(columns), 'features:', round(-scores.mean(), 4))
```

0.8187, 0.9255, 0.9176. One feature wins again, on a stock the lecture never mentioned.

</details>

---

### I3 · Refit and score once  ★★★☆☆

Refit the winning candidate on **all** of `m_train` and score it once on `m_test`.

In [ ]:
msft = pd.DataFrame({
    'vol_20d': rets['MSFT'].rolling(20).std(),
    'vol_60d': rets['MSFT'].rolling(60).std(),
    'ret_20d': rets['MSFT'].rolling(20).mean(),
})
msft['vol_next'] = rets['MSFT'].rolling(20).std().shift(-20)
msft = msft.dropna()
m_train = msft.loc[:'2022-12-31']
m_test = msft.loc['2023-01-01':]

final = ...
m_rmse = ...
print(m_rmse)

<details>
<summary>💡 Hint</summary>

Fit `LinearRegression()` on `m_train[A]`, predict on `m_test[A]`, take the root mean squared error.

</details>

<details>
<summary>✅ Solution</summary>

```python
msft = pd.DataFrame({
    'vol_20d': rets['MSFT'].rolling(20).std(),
    'vol_60d': rets['MSFT'].rolling(60).std(),
    'ret_20d': rets['MSFT'].rolling(20).mean(),
})
msft['vol_next'] = rets['MSFT'].rolling(20).std().shift(-20)
msft = msft.dropna()
m_train = msft.loc[:'2022-12-31']
m_test = msft.loc['2023-01-01':]

final = LinearRegression()
final.fit(m_train[A], m_train['vol_next'])
m_rmse = np.sqrt(mean_squared_error(m_test['vol_next'], final.predict(m_test[A])))
print(m_rmse)
```

0.3486. The folds were only ever a way of choosing. Once the choice is made the model may use every training row available.

</details>

---

### I4 · Against the two baselines  ★★★☆☆

Score the average and the persistence rule on `m_test` too, and print all three in order.

In [ ]:
msft = pd.DataFrame({
    'vol_20d': rets['MSFT'].rolling(20).std(),
    'vol_60d': rets['MSFT'].rolling(60).std(),
    'ret_20d': rets['MSFT'].rolling(20).mean(),
})
msft['vol_next'] = rets['MSFT'].rolling(20).std().shift(-20)
msft = msft.dropna()
m_train = msft.loc[:'2022-12-31']
m_test = msft.loc['2023-01-01':]

results = {
    'model': ...,
    'average': ...,
    'persistence': ...,
}

# print them smallest first
for name in results:
    ...

<details>
<summary>💡 Hint</summary>

This is exercise C5 with `m_train` and `m_test` in place of `train` and `test`.

</details>

<details>
<summary>✅ Solution</summary>

```python
msft = pd.DataFrame({
    'vol_20d': rets['MSFT'].rolling(20).std(),
    'vol_60d': rets['MSFT'].rolling(60).std(),
    'ret_20d': rets['MSFT'].rolling(20).mean(),
})
msft['vol_next'] = rets['MSFT'].rolling(20).std().shift(-20)
msft = msft.dropna()
m_train = msft.loc[:'2022-12-31']
m_test = msft.loc['2023-01-01':]

model = LinearRegression().fit(m_train[A], m_train['vol_next'])

results = {
    'model': np.sqrt(mean_squared_error(m_test['vol_next'], model.predict(m_test[A]))),
    'average': np.sqrt(mean_squared_error(m_test['vol_next'], np.full(len(m_test), m_train['vol_next'].mean()))),
    'persistence': np.sqrt(mean_squared_error(m_test['vol_next'], m_test['vol_20d'])),
}

for name in sorted(results, key=results.get):
    print(f'{name:12} {results[name]:.4f}')
```

model 0.3486, persistence 0.4050, average 0.3931. The same ordering as Apple, on a stock chosen without looking first.

</details>

---

### I5 · And on a quiet stock  ★★★★☆

Run the whole thing once more on `KO`: build the table, cross-validate the three candidates, then report the test RMSE of the winner next to persistence.

In [ ]:
ko = ...

ko_train = ...
ko_test = ...

# cross-validate, then refit the winner and score once
...

<details>
<summary>💡 Hint 1</summary>

Everything you need is in I1 to I4. Change the ticker and nothing else.

</details>

<details>
<summary>💡 Hint 2</summary>

Coca-Cola is a much calmer stock, so expect every number to be smaller. What matters is whether the **ordering** survives.

</details>

<details>
<summary>✅ Solution</summary>

```python
ko = pd.DataFrame({
    'vol_20d': rets['KO'].rolling(20).std(),
    'vol_60d': rets['KO'].rolling(60).std(),
    'ret_20d': rets['KO'].rolling(20).mean(),
})
ko['vol_next'] = rets['KO'].rolling(20).std().shift(-20)
ko = ko.dropna()

ko_train = ko.loc[:'2022-12-31']
ko_test = ko.loc['2023-01-01':]

folds = TimeSeriesSplit(n_splits=5)
for columns in [A, B, C]:
    scores = cross_val_score(LinearRegression(), ko_train[columns], ko_train['vol_next'],
                             cv=folds, scoring='neg_root_mean_squared_error')
    print(len(columns), 'features, CV:', round(-scores.mean(), 4))

model = LinearRegression().fit(ko_train[A], ko_train['vol_next'])
print('test RMSE  :', round(np.sqrt(mean_squared_error(ko_test['vol_next'], model.predict(ko_test[A]))), 4))
print('persistence:', round(np.sqrt(mean_squared_error(ko_test['vol_next'], ko_test['vol_20d'])), 4))
```

Cross-validation picks one feature again (0.5312 against 0.5478), and the model beats persistence 0.2690 to 0.3055. Every number is smaller than Apple's because Coca-Cola moves less, which is why an RMSE always has to be read next to a baseline from the same data.

</details>

---

## 🔁 K · Everything you already had

Loops, functions, dictionaries and figures, pointed at the workflow you have just learned.

### K1 · One function, any ticker  ★★★☆☆  · revisits S2

Write `evaluate(ticker)`: it builds the table for that ticker, splits it at the end of 2022, fits `A` on the training rows, and returns the test RMSE.

In [ ]:
def evaluate(ticker):
    ...

print('AAPL:', ...)
print('NVDA:', ...)

<details>
<summary>💡 Hint 1</summary>

The body is exercise I3 with `ticker` in place of `'MSFT'`.

</details>

<details>
<summary>💡 Hint 2</summary>

Only two columns are needed: `vol_20d` and `vol_next`.

</details>

<details>
<summary>✅ Solution</summary>

```python
def evaluate(ticker):
    frame = pd.DataFrame({'vol_20d': rets[ticker].rolling(20).std()})
    frame['vol_next'] = rets[ticker].rolling(20).std().shift(-20)
    frame = frame.dropna()

    tr = frame.loc[:'2022-12-31']
    te = frame.loc['2023-01-01':]

    model = LinearRegression()
    model.fit(tr[['vol_20d']], tr['vol_next'])
    return np.sqrt(mean_squared_error(te['vol_next'], model.predict(te[['vol_20d']])))

print('AAPL:', round(evaluate('AAPL'), 4))
print('NVDA:', round(evaluate('NVDA'), 4))
```

Apple 0.4056, Nvidia 1.1423. Apple's number differs slightly from the 0.4056 earlier in the notebook, because dropping `vol_60d` also drops the rows where a sixty-day window had not yet filled. A function is the right shape here because the next exercise wants this done eleven times.

</details>

---

### K2 · Does it beat persistence everywhere  ★★★★☆  · revisits S2

For every ticker in `rets`, compute the model's test RMSE and the persistence rule's test RMSE. Count how many of the eleven the model wins.

In [ ]:
results = {}

for ticker in rets.columns:
    ...

wins = ...
print('model wins on', ..., 'of', ...)

<details>
<summary>💡 Hint 1</summary>

Store a pair in the dictionary: `results[ticker] = (model_rmse, pers_rmse)`.

</details>

<details>
<summary>💡 Hint 2</summary>

`sum(1 for t in results if results[t][0] < results[t][1])` counts the wins.

</details>

<details>
<summary>✅ Solution</summary>

```python
results = {}

for ticker in rets.columns:
    frame = pd.DataFrame({'vol_20d': rets[ticker].rolling(20).std()})
    frame['vol_next'] = rets[ticker].rolling(20).std().shift(-20)
    frame = frame.dropna()

    tr = frame.loc[:'2022-12-31']
    te = frame.loc['2023-01-01':]

    model = LinearRegression()
    model.fit(tr[['vol_20d']], tr['vol_next'])
    model_rmse = np.sqrt(mean_squared_error(te['vol_next'], model.predict(te[['vol_20d']])))
    pers_rmse = np.sqrt(mean_squared_error(te['vol_next'], te['vol_20d']))
    results[ticker] = (model_rmse, pers_rmse)

wins = sum(1 for t in results if results[t][0] < results[t][1])
print('model wins on', wins, 'of', len(results))
```

11 out of 11. A rule chosen on one stock, with the choice made without ever looking at 2023 or 2024, transfers to every other name in the universe. That is a far stronger result than a single number on a single stock.

</details>

---

### K3 · Draw the eleven  ★★★☆☆  · revisits S3

Draw a bar chart of the improvement, `persistence - model`, for the eleven tickers, sorted from largest to smallest.

In [ ]:
results = {}
for ticker in rets.columns:
    ...

improvement = ...

fig, ax = plt.subplots(figsize=(8, 3.2))
...
plt.show()

<details>
<summary>💡 Hint 1</summary>

Reuse the loop from K2, then `pd.Series(improvement).sort_values(ascending=False)`.

</details>

<details>
<summary>💡 Hint 2</summary>

`ax.bar(ranked.index, ranked.values)` draws it, and `ax.axhline(0, color='black')` marks the line a bar has to clear.

</details>

<details>
<summary>✅ Solution</summary>

```python
improvement = {}

for ticker in rets.columns:
    frame = pd.DataFrame({'vol_20d': rets[ticker].rolling(20).std()})
    frame['vol_next'] = rets[ticker].rolling(20).std().shift(-20)
    frame = frame.dropna()

    tr = frame.loc[:'2022-12-31']
    te = frame.loc['2023-01-01':]

    model = LinearRegression().fit(tr[['vol_20d']], tr['vol_next'])
    model_rmse = np.sqrt(mean_squared_error(te['vol_next'], model.predict(te[['vol_20d']])))
    pers_rmse = np.sqrt(mean_squared_error(te['vol_next'], te['vol_20d']))
    improvement[ticker] = pers_rmse - model_rmse

ranked = pd.Series(improvement).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 3.2))
ax.bar(ranked.index, ranked.values)
ax.axhline(0, color='black', linewidth=1)
ax.set_ylabel('RMSE saved (percentage points)')
ax.set_title('How much the model beats persistence by, 2023 to 2024', loc='left')
plt.show()
```

Every bar is above the line, and JNJ gains the most in relative terms. The stocks that move most are the ones where a forecast is worth the most, which is convenient rather than surprising.

</details>

---

### K4 · Say it in one sentence  ★★★☆☆  · revisits S1

Print a single line reporting the chosen model, its test RMSE and the persistence RMSE, each to three decimals, using an f-string.

In [ ]:
model = LinearRegression()
model.fit(train[A], train['vol_next'])
model_rmse = np.sqrt(mean_squared_error(test['vol_next'], model.predict(test[A])))
pers_rmse = np.sqrt(mean_squared_error(test['vol_next'], test['vol_20d']))

sentence = ...
print(sentence)

<details>
<summary>💡 Hint</summary>

`f'...{value:.3f}...'` rounds inside the string. Name the feature set, both numbers, and the years the test block covers.

</details>

<details>
<summary>✅ Solution</summary>

```python
model = LinearRegression()
model.fit(train[A], train['vol_next'])
model_rmse = np.sqrt(mean_squared_error(test['vol_next'], model.predict(test[A])))
pers_rmse = np.sqrt(mean_squared_error(test['vol_next'], test['vol_20d']))

sentence = (f'A linear regression on vol_20d scores {model_rmse:.3f} on 2023 to 2024, '
            f'against {pers_rmse:.3f} for repeating last month.')
print(sentence)
```

One sentence with a model, a number and a comparison. That is the smallest honest unit of a result, and a surprising amount of published work never gets as far as the comparison.

</details>

---

### K5 · The whole workflow, as one table  ★★★★★  · revisits S2

Build a DataFrame with one row per ticker and three columns: the cross-validation error of the chosen candidate on the training block, the test RMSE, and the persistence RMSE. Sort it by test RMSE.

In [ ]:
rows = []

for ticker in rets.columns:
    ...

summary = ...
summary

<details>
<summary>💡 Hint 1</summary>

Per ticker: build the table, split it, cross-validate `A` on the training block, refit on all of it, then score both rules on the test block.

</details>

<details>
<summary>💡 Hint 2</summary>

Append a dictionary per ticker and finish with `pd.DataFrame(rows).set_index('ticker').sort_values('test')`.

</details>

<details>
<summary>✅ Solution</summary>

```python
rows = []

for ticker in rets.columns:
    frame = pd.DataFrame({'vol_20d': rets[ticker].rolling(20).std()})
    frame['vol_next'] = rets[ticker].rolling(20).std().shift(-20)
    frame = frame.dropna()

    tr = frame.loc[:'2022-12-31']
    te = frame.loc['2023-01-01':]

    cv = cross_val_score(LinearRegression(), tr[['vol_20d']], tr['vol_next'],
                         cv=TimeSeriesSplit(n_splits=5),
                         scoring='neg_root_mean_squared_error')

    model = LinearRegression().fit(tr[['vol_20d']], tr['vol_next'])

    rows.append({
        'ticker': ticker,
        'cv': -cv.mean(),
        'test': np.sqrt(mean_squared_error(te['vol_next'], model.predict(te[['vol_20d']]))),
        'persistence': np.sqrt(mean_squared_error(te['vol_next'], te['vol_20d'])),
    })

summary = pd.DataFrame(rows).set_index('ticker').sort_values('test')
summary
```

The `cv` column is always larger than the `test` column, because each fold fits on fewer rows and scores on a harder stretch of history. That gap is normal and worth expecting: cross-validation is a conservative estimate, not a prediction of the test score.

This table is the entire session in one object. If you can produce it from a blank cell, you can do everything Session 5 asked of you.

</details>

---

### K6 · Calm days and busy days  ★★★★☆  · revisits S3

Split the test block in two with a **boolean mask**: days whose `vol_20d` is below its median, and the rest. Score the model separately on each half.

Is the forecast equally good in quiet markets and busy ones?

In [ ]:
model = LinearRegression()
model.fit(train[['vol_20d']], train['vol_next'])
predictions = model.predict(test[['vol_20d']])
errors = test['vol_next'] - predictions

calm = ...

print('calm days:', ...)
print('busy days:', ...)

<details>
<summary>💡 Hint 1</summary>

`calm = test['vol_20d'] < test['vol_20d'].median()` is a column of True and False, one per test day.

</details>

<details>
<summary>💡 Hint 2</summary>

`errors[calm]` keeps the calm days and `errors[~calm]` keeps the rest. The `~` flips a mask, exactly as in Session 4's confusion matrix.

</details>

<details>
<summary>✅ Solution</summary>

```python
model = LinearRegression()
model.fit(train[['vol_20d']], train['vol_next'])
predictions = model.predict(test[['vol_20d']])
errors = test['vol_next'] - predictions

calm = test['vol_20d'] < test['vol_20d'].median()

print('calm days:', round(np.sqrt((errors[calm] ** 2).mean()), 4))
print('busy days:', round(np.sqrt((errors[~calm] ** 2).mean()), 4))
```

0.3784 on the 241 calmest test days against 0.4311 on the rest, so the forecast is about 14% worse when the market is busy.

That is the direction you would expect and it is worth stating, because the busy half is the half a risk model exists for. A single RMSE averages the two together and hides it.

Two masks, a `~`, and no loop anywhere. Everything in this exercise is Session 3 arithmetic pointed at a Session 5 question.

</details>

---

## 🏁 Done

You have fitted a model, scored it against two rules that use no model at all, watched a single validation split give two different answers, cross-validated with folds that follow the calendar, and run the whole thing on stocks the lecture never mentioned.

The one habit worth carrying out of this notebook: no score means anything until it sits next to a baseline and a statement of what was allowed to see what.

**Next:** open `session_05_case.ipynb` (*The Analyst's Notebook, Part 5*), where the risk report finally gets a model and has to justify it.

*Stuck for more than 15 minutes on anything? Ask a friend, ask an AI for a hint (not the answer), or email me at `jobo@econ.au.dk`.*